<!-- 5 Tank Recycle Case -->

3 Tank Recycle Scenario

In [ ]:
original_dir = pwd()
cd(original_dir)
include("3 Tank dSin Bioreactor.jl")
include("MPC Model 3tk dSin Bioreactor.jl")

create_mpc_model (generic function with 1 method)

In [2]:
##############################
# ASSIGN WEIGHTS 
##############################
#alpha_tk = assign_weights()
alpha_tk = nothing 
##############################
# CREATE MPC MODEL
##############################
mpc_model = create_mpc_model(alpha_tk)

println(alpha_tk)

nothing


In [3]:
##############################
# RUN MPC
##############################
using JuMP

n_steps = Int(nt/dt)+1     

Vprofile = zeros(n_steps,NI)  
Ssprofile = zeros(n_steps, 1)
Xbprofile = zeros(n_steps, 1)
Soprofile = zeros(n_steps, 1)

# INPUT PROFILES 
Klaprofile = zeros(n_steps-1, 1)
F_profile =  zeros(n_steps-1, O)

# STATE INITIAL CONDITIONS 
Vprofile[1, :] = V0
Ssprofile[1] = Ss0
Xbprofile[1] = Xb0
Soprofile[1] = So0



for i = 2:n_steps

    fmax = fmax_vec[:, i]
    Bd = Bd_vec[:, i]
    Ss_in_max = Ss_in_vec[i]

    # FIX INITIAL CONDITIONS IN MODEL AND RUN 
    fix.(mpc_model[:V][:, 1], V0, force=true)
    fix.(mpc_model[:Ss][1, 1], Ss0, force=true)
    fix.(mpc_model[:Xb][1, 1], Xb0, force=true)
    fix.(mpc_model[:So][1, 1], So0, force=true)

    # FIX DISTURBANCES IN MODEL
    fix.(mpc_model[:fmax], fmax, force=true)
    fix.(mpc_model[:Bd], Bd, force=true)
    fix.(mpc_model[:Ss_in], Ss_in_max, force=true)

    # SOLVE MODEL 
    optimize!(mpc_model)
    status = JuMP.termination_status(mpc_model)

    if status != MOI.LOCALLY_SOLVED && status != MOI.OPTIMAL
        println("Model did not converge to global or local optima")
    else
        println("Model solved successfully")
    end
    
    println(status)
    
    # SAVE RESULTS FOR STATES 
    # SAVE RESULTS FROM 2ND TIME STEP (FIRST TIME STEP IS INITIAL CONDITION)
    V0 = [value(mpc_model[:V][ni, 2]) for ni in 1:NI]
    Ss0 = value(mpc_model[:Ss][1, 2])
    Xb0 = value(mpc_model[:Xb][1, 2])
    So0 = value(mpc_model[:So][1, 2])

    # SAVE RESULTS FOR INPUTS
    # SAVE RESULTS FROM 1ST TIME STEP 
    Kla0 = value(mpc_model[:Kla][1])
    f0 = value.(mpc_model[:f][:, 1])
    
    # SAVE RESULTS FOR STATES
    Vprofile[i, :] = V0 
    Ssprofile[i] = Ss0 
    Xbprofile[i] = Xb0 
    Soprofile[i] = So0 

    # SAVE RESULTS FOR INPUTS 
    Klaprofile[i-1] = Kla0 
    F_profile[i-1, :] = f0



end 






******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVE